In [1]:
# Initialize gee
import geemap
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [2]:
# Carregar camada
path_json = "../outputs/bacias_meso_SF.geojson"
geom_ee = geemap.geojson_to_ee(path_json)
area = geom_ee.geometry()

In [3]:
# 3. Carregar a coleção MODIS (Reflectância de Superfície)
modis = ee.ImageCollection('MODIS/061/MOD09A1') \
    .filterBounds(area) \
    .filterDate('2020-01-01', '2020-12-31')

In [4]:
# 4. Função para calcular MSAVI e Albedo
def add_indices(image):
    # Aplicar fator de escala do MODIS para obter a reflectância real [7]
    img_scaled = image.multiply(0.0001) 
    
    # Cálculo do MSAVI usando .expression() [4]
    msavi = img_scaled.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': img_scaled.select('sur_refl_b02'), # Banda NIR no MODIS
            'RED': img_scaled.select('sur_refl_b01')  # Banda RED no MODIS
        }).rename('MSAVI')
        
    # Cálculo do Albedo (Exemplo usando fórmula empírica comum para MODIS)
    # Verifique os coeficientes exatos da metodologia que você está seguindo
    albedo = img_scaled.expression(
        '0.160 * B1 + 0.291 * B2 + 0.243 * B3 + 0.116 * B4 + 0.112 * B5 + 0.081 * B7 - 0.0015', {
            'B1': img_scaled.select('sur_refl_b01'),
            'B2': img_scaled.select('sur_refl_b02'),
            'B3': img_scaled.select('sur_refl_b03'),
            'B4': img_scaled.select('sur_refl_b04'),
            'B5': img_scaled.select('sur_refl_b05'),
            'B7': img_scaled.select('sur_refl_b07')
        }).rename('Albedo')

    # Adiciona as novas bandas à imagem original e mantém as propriedades de data [4, 8]
    return image.addBands([msavi, albedo]).copyProperties(image, ['system:time_start'])

# 1. Função para extrair a média e a data de cada imagem
def extrair_serie(image):
    # Calcula a média do MSAVI e Albedo dentro do seu ROI
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area,
        scale=500, # Resolução nativa do MODIS em metros
        maxPixels=1e9
    )
    
    # Retorna os dados como atributos de uma Feature (sem a geometria pesada)
    return ee.Feature(None, {
        'Data': image.date().format('YYYY-MM-dd'),
        'MSAVI': stats.get('MSAVI'),
        'Albedo': stats.get('Albedo')
    })

In [7]:
# 5. Mapear a função sobre toda a coleção de imagens
modis_com_indices = modis.map(add_indices)

In [4]:
# 2. Mapeia a função sobre a coleção que criamos no passo anterior
serie_temporal_fc = modis_com_indices.map(extrair_serie)

# 3. Transforma a FeatureCollection do Earth Engine direto para um DataFrame do Pandas
df = geemap.ee_to_df(ee.FeatureCollection(serie_temporal_fc))

# 4. Limpa e organiza a tabela
df = df.dropna().sort_values('Data')
df.to_excel("../data/processed/msavi-albedo.xlsx", index='Data')
print(df.head())

NameError: name 'modis_com_indices' is not defined

# Testar a performance no XEE

In [8]:
# 1. Inicializar o XEE
import xarray as xr
from xee import helpers
import geopandas as gpd


# Lê o arquivo GeoJSON localmente para o ambiente Python
gdf = gpd.read_file(path_json)

# Extrai a geometria do GeoPandas
aoi = gdf.geometry.union_all()

# Definir parâmetros para fit da geometria
GRID_CRS = "+proj=cea +lon_0=0 +lat_ts=0 +datum=WGS84 +units=m +no_defs"
AOI_CRS = "+proj=longlat +datum=WGS84 +no_defs"
GRID_SCALE = (1000, -1000)

grid_params = helpers.fit_geometry(
    geometry=aoi,
    geometry_crs=AOI_CRS,
    grid_crs=GRID_CRS,
    grid_scale=GRID_SCALE,
)

# Filtrar o a feature collection apenas com as bandas de interesse (MSAVI e Albedo)
modis_com_indices = modis_com_indices.select(["MSAVI", "Albedo"])

# Abre a coleção do GEE como um cubo de dados multidimensional
ds = xr.open_dataset(
    modis_com_indices,
    engine='ee',
    **grid_params,
    chunks={"time": 1, "y": 512, "x": 512},
)
ds

C:\Users\Gab\AppData\Local\Temp\ipykernel_40580\1294267440.py:29: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


<xarray.Dataset> Size: 680MB
Dimensions:  (time: 46, y: 1464, x: 1263)
Coordinates:
  * time     (time) datetime64[ns] 368B 2020-01-01 2020-01-09 ... 2020-12-26
  * y        (y) float64 12kB -8.015e+05 -8.025e+05 ... -2.264e+06 -2.264e+06
  * x        (x) float64 10kB -5.304e+06 -5.302e+06 ... -4.042e+06 -4.042e+06
Data variables:
    MSAVI    (time, y, x) float32 340MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    Albedo   (time, y, x) float32 340MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>

In [10]:
ds=ds.sortby('time')*1

In [11]:
ds_monthly = ds.resample(time='ME').mean('time')
ds_monthly

<xarray.Dataset> Size: 178MB
Dimensions:  (time: 12, y: 1464, x: 1263)
Coordinates:
  * time     (time) datetime64[ns] 96B 2020-01-31 2020-02-29 ... 2020-12-31
  * y        (y) float64 12kB -8.015e+05 -8.025e+05 ... -2.264e+06 -2.264e+06
  * x        (x) float64 10kB -5.304e+06 -5.302e+06 ... -4.042e+06 -4.042e+06
Data variables:
    MSAVI    (time, y, x) float32 89MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    Albedo   (time, y, x) float32 89MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>

In [ ]:
ds_monthly.sel(time = '2020').MSAVI.plot(x='lon', y='lat', col='time', robust=True, col_wrap = 6, cmap='GrYlOrRed')

In [ ]:
# 2. Prepara o cálculo da média espacial do MSAVI e Albedo
# O Dask organiza as requisições para rodarem em paralelo no servidor do Google
serie_media = ds[['MSAVI', 'Albedo']].mean(dim=['lon', 'lat'])

In [ ]:
# 3. O comando .to_dataframe() "força" a execução (Lazy Loading) e baixa a tabela final
df_xee = serie_media.to_dataframe().dropna()
print(df_xee.head())